# Vietnamese Medical Visual Question Answering (VQA)
## Modular Architecture with Cross-Modal Co-Attention and LLM-Enhanced Refinement

### Experimental Setup: 5 Research Configurations

| Session | Variant | Goal | Strategy |
|:---:|:---:|:---|:---|
| 1 | **A1** | Modular Baseline | DenseNet-121 (XRV) + PhoBERT + LSTM |
| 2 | **A2** | Modular Advanced | DenseNet-121 (XRV) + PhoBERT + Transformer |
| 3 | **B1** | Foundation Baseline | LLaVA-Med-7B (Zero-shot Evaluation) |
| 4 | **B2** | Domain Adaptation | LLaVA-Med-7B + QLoRA Fine-tuning |
| 5 | **DPO** | Clinical Alignment | LLaVA-Med (B2) + Direct Preference Optimization |


## 0. Project Repository Initialization
Clone the project repository from GitHub and configure the environment for modular access.

In [ ]:
import os
import sys

# 1. CONFIGURATION:
REPO_URL = "https://github.com/QuangVoAI/DL_MedicalVQA_Project.git"
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

# 2. CLONING
if not os.path.exists(REPO_NAME):
    print(f"Cloning repository: {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Repository {REPO_NAME} already exists. Pulling latest changes...")
    !cd {REPO_NAME} && git pull

# 3. WORKSPACE SETUP
os.chdir(os.path.join(os.getcwd(), REPO_NAME))

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
    
print(f"Successfully initialized workspace at: {os.getcwd()}")

## 1. Environment Configuration and Workspace Setup
Installation of necessary dependencies and configuration of the Python path for modular script access.

In [ ]:
import torch
import yaml
import pandas as pd
from datasets import load_dataset

# Install dependencies (Quiet mode)
!pip install -q -r requirements.txt
!pip install -q bert-score rouge-score sentence-transformers

# Hardware Acceleration Detection
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() 
    else 'mps' if torch.backends.mps.is_available() 
    else 'cpu'
)

print(f"Device: {DEVICE}")
print(f"PyTorch Version: {torch.__version__}")

## 2. Dataset Integration from Hugging Face Hub
Loading the refined Vietnamese Medical VQA dataset directly from the remote repository.

In [ ]:
REPO_ID = "SpringWang08/medical-vqa-vi"

print(f"Synchronizing dataset with remote repository: {REPO_ID}...")
hf_dataset = load_dataset(REPO_ID)

print("\nDataset Split Statistics:")
for split in hf_dataset.keys():
    print(f" - {split.capitalize()}: {len(hf_dataset[split])} samples")

## 3. Global Hyperparameter and System Configuration
Parsing the YAML configuration file to initialize model dimensions, training schedules, and evaluation parameters.

In [ ]:
with open('configs/medical_vqa.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print("System Parameters Initialized:")
print(f" - Image Size: {cfg['data']['image_size']}")
print(f" - Text Embedding: {cfg['model_a']['phobert_model']}")
print(f" - Batch Size: {cfg['train']['batch_size']}")

## 4. Data Pipeline and Tokenization
Implementation of custom PyTorch DataLoaders with **Medical-specific normalization** and PhoBERT tokenization.

In [ ]:
from transformers import AutoTokenizer
from src.data.medical_dataset import MedicalVQADataset
from torch.utils.data import DataLoader
from src.utils.visualization import MedicalImageTransform

tokenizer = AutoTokenizer.from_pretrained(cfg['model_a']['phobert_model'])

# [FIX] Use Medical-specific transform instead of ImageNet stats
transform = MedicalImageTransform(size=cfg['data']['image_size'])

train_ds = MedicalVQADataset(hf_dataset=hf_dataset['train'], tokenizer=tokenizer, transform=transform)
val_ds = MedicalVQADataset(hf_dataset=hf_dataset['validation'], tokenizer=tokenizer, transform=transform)
test_ds = MedicalVQADataset(hf_dataset=hf_dataset['test'], tokenizer=tokenizer, transform=transform)

train_loader = DataLoader(train_ds, batch_size=cfg['train']['batch_size'], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg['train']['batch_size'], shuffle=False)
test_loader = DataLoader(test_ds, batch_size=cfg['train']['batch_size'], shuffle=False)

print(f"DataLoaders initialized with Medical CLAHE + Normalize.")

## 5. Implementation of Variant A (Modular Architectures)
Construction of the vision-language fusion model using DenseNet-121 XRV and PhoBERT.

In [ ]:
from src.models.medical_vqa_model import MedicalVQAModelA

# [FIXED] Proper keyword argument propagation
model_A1 = MedicalVQAModelA(
    decoder_type='lstm', 
    vocab_size=len(tokenizer),
    hidden_size=cfg['model_a']['hidden_size'],
    phobert_model=cfg['model_a']['phobert_model']
).to(DEVICE)

model_A2 = MedicalVQAModelA(
    decoder_type='transformer', 
    vocab_size=len(tokenizer),
    hidden_size=cfg['model_a']['hidden_size'],
    phobert_model=cfg['model_a']['phobert_model']
).to(DEVICE)

print(f"Variant A1 & A2 models initialized.")

## 2. Experimental Execution
Run the following sessions sequentially or independently to obtain results for the report.

### Session 1: Training Variant A1 (LSTM Baseline)
Modular architecture with traditional recurrent decoding.

In [ ]:
!python train_medical.py --config configs/medical_vqa.yaml --variant A1

### Session 2: Training Variant A2 (Transformer Advanced)
Modular architecture with attention-based transformer decoding.

In [ ]:
!python train_medical.py --config configs/medical_vqa.yaml --variant A2

### Session 3: Evaluation of Variant B1 (Zero-shot Foundation)
Direct inference on LLaVA-Med-7B without domain-specific training.

In [ ]:
!python train_medical.py --config configs/medical_vqa.yaml --variant B1

### Session 4: Training Variant B2 (LLaVA-Med Fine-tuning)
Domain adaptation using QLoRA 4-bit supervised fine-tuning.

In [ ]:
!python train_medical.py --config configs/medical_vqa.yaml --variant B2

### Session 5: Training Variant DPO (Clinical Safety Alignment)
Direct Preference Optimization to reduce medical hallucinations.

In [ ]:
!python train_medical.py --config configs/medical_vqa.yaml --variant DPO

## 3. Final Comparative Evaluation
Aggregate results across all 5 configurations (A1, A2, B1, B2, DPO).

In [ ]:
from src.engine.medical_eval import MedicalVQAEvaluator
evaluator = MedicalVQAEvaluator(device=DEVICE, tokenizer=tokenizer)

models_to_eval = ["A1", "A2", "B1", "B2", "DPO"]
for variant in models_to_eval:
    print(f"\n--- Evaluating {variant} ---")
    # Note: In actual notebook, you would load the checkpoints first
    # results = evaluator.evaluate(model, test_loader, variant_type='A' if 'A' in variant else 'B')
    pass

print("Evaluation framework ready for comparative analysis.")

## 4. Human Evaluation Protocol (Academic Requirement)
Compare Model B2 (SFT) vs Model DPO (Aligned) through manual clinician-style review.

In [ ]:
import random

def manual_review(samples, preds_b2, preds_dpo):
    """
    Compare SFT (B2) vs DPO. Record preference based on medical correctness.
    """
    results = {"B2_wins": 0, "DPO_wins": 0, "Tie": 0}
    # Logic for displaying image and capturing input
    print("Manual review session initialized.")
    return results

## 5. Clinical Conclusion and Scientific Documentation
**References:**
- LLaVA-Med: Li et al., arXiv 2023.
- DPO: Rafailov et al., NeurIPS 2023.